# 🚀 Entrenamiento de Modelos de IA en Google Colab

Este cuaderno te permite entrenar los modelos **ResNet50** y **MobileNetV2** utilizando la GPU gratuita de Google Colab. 

### ⚠️ IMPORTANTE: Configurar Entorno de Ejecución con GPU
Antes de ejecutar cualquier celda, activa la aceleración por hardware:
1. Ve al menú superior: **Entorno de ejecución** (Runtime) -> **Cambiar tipo de entorno de ejecución** (Change runtime type).
2. En **Acelerador de hardware** (Hardware accelerator), selecciona **T4 GPU** (o cualquier GPU disponible).
3. Haz clic en **Guardar** (Save).

In [ ]:
# Verificar que la GPU esté activa
import torch
print("¿GPU Disponible?:", "Sí" if torch.cuda.is_available() else "No")
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))

## 📦 1. Cargar el Dataset

Para entrenar el modelo, necesitas subir tus imágenes. Sigue estos pasos:
1. En tu computadora, ve a la carpeta `microservicio/dataset`.
2. Comprime la carpeta `processed` en un archivo ZIP llamado `dataset.zip`.
3. Ejecuta la celda de abajo para habilitar el selector de archivos y sube tu `dataset.zip`.

In [ ]:
from google.colab import files
import os
import zipfile
import shutil

print("Por favor, selecciona y sube tu archivo 'dataset.zip'...")
uploaded = files.upload()

if 'dataset.zip' in uploaded:
    # Limpiar carpeta si ya existía de un intento anterior
    if os.path.exists('dataset_temp'):
        shutil.rmtree('dataset_temp')
        
    print("Extracting dataset...")
    # Extractor robusto compatible con Windows ZIP (convierte \ a / en Linux)
    with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
        for member in zip_ref.namelist():
            # Reemplazar separadores de Windows por los de Linux
            clean_name = member.replace('\\', '/')
            dest_path = os.path.join('dataset_temp', clean_name)
            
            # Crear carpetas si no existen
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            
            # Escribir archivo físico si no es un directorio
            if not clean_name.endswith('/'):
                with zip_ref.open(member) as source, open(dest_path, 'wb') as target:
                    shutil.copyfileobj(source, target)
                    
    print("¡Dataset extraído correctamente!")
    
    # Buscar de forma recursiva dónde están las carpetas de las clases
    clases_objetivo = ['COLISION_FRONTAL', 'COLISION_LATERAL', 'COLISION_TRASERA', 'VOLCADURA', 'INCENDIO_VEHICULAR', 'CRISTAL_ROTO', 'SIN_DANO_VISIBLE']
    found_path = None
    for root, dirs, files_list in os.walk('dataset_temp'):
        dirs_upper = [d.upper() for d in dirs]
        if any(c in dirs_upper for c in clases_objetivo):
            found_path = root
            break
            
    if found_path:
        print(f"📂 Dataset localizado con éxito en: {found_path}")
        print("Carpetas encontradas:", os.listdir(found_path))
    else:
        print("⚠️ Alerta: No se encontraron las carpetas de las clases. Estructura de la carpeta extraída:")
        for root, dirs, files_in_dir in os.walk('dataset_temp'):
            print(f" - {root} (subdirs: {dirs}, files: {len(files_in_dir)})")
else:
    print("Error: No se subió un archivo llamado 'dataset.zip'")

## ⚙️ 2. Instalar Librerías Adicionales
Instalamos los requisitos necesarios para el entrenamiento y métricas.

In [ ]:
!pip install torchvision pillow numpy pandas scikit-learn matplotlib tqdm

## 🧠 3. Código de Entrenamiento (PyTorch)
Definimos el pipeline de entrenamiento idéntico al de tu servidor local pero optimizado para GPU CUDA.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

CLASSES = {
    'COLISION_FRONTAL': 0,
    'COLISION_LATERAL': 1,
    'COLISION_TRASERA': 2,
    'VOLCADURA': 3,
    'INCENDIO_VEHICULAR': 4,
    'CRISTAL_ROTO': 5,
    'SIN_DANO_VISIBLE': 6,
}
CLASS_NAMES = {v: k for k, v in CLASSES.items()}

def get_dataset_directory():
    if not os.path.exists('dataset_temp'):
        print('❌ ERROR: La carpeta dataset_temp no existe en esta sesión.')
        return None
        
    clases_objetivo = list(CLASSES.keys())
    for root, dirs, _ in os.walk('dataset_temp'):
        dirs_upper = [d.upper() for d in dirs]
        if any(c in dirs_upper for c in clases_objetivo):
            return Path(root)
            
    print('❌ ERROR: No se encontraron las carpetas de las clases dentro de dataset_temp.')
    print('Estructura de archivos extraída detectada:')
    for root, dirs, files_in_dir in os.walk('dataset_temp'):
        level = root.replace('dataset_temp', '').count(os.sep)
        indent = ' ' * 4 * level
        folder_name = os.path.basename(root) or 'dataset_temp'
        print(f'{indent}📁 {folder_name}/')
        subindent = ' ' * 4 * (level + 1)
        for d in dirs:
            print(f'{subindent}📁 {d}/')
        for f in files_in_dir[:3]:
            print(f'{subindent}📄 {f}')
    return None

MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

class IncidentDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label, str(img_path)
        except Exception as e:
            return torch.zeros(3, 224, 224), label, str(img_path)

def load_dataset(dataset_dir):
    print(f'📂 Cargando dataset desde: {dataset_dir}')
    image_paths, labels = [], []
    
    carpetas_existentes = {d.upper(): d for d in os.listdir(dataset_dir) if os.path.isdir(dataset_dir / d)}
    
    for class_name, class_idx in CLASSES.items():
        real_folder_name = carpetas_existentes.get(class_name, class_name)
        class_dir = dataset_dir / real_folder_name
        
        if not class_dir.exists():
            print(f'  ⚠️ Advertencia: No existe la carpeta {class_name}')
            continue
        image_files = [f for f in class_dir.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
        print(f'  ✅ {class_name}: {len(image_files)} imágenes')
        for img_file in image_files:
            image_paths.append(img_file)
            labels.append(class_idx)
    return image_paths, labels

def create_model(model_type="resnet50", num_classes=7):
    print(f'🧠 Creando modelo {model_type}...')
    if model_type == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        for param in model.layer1.parameters(): param.requires_grad = False
        for param in model.layer2.parameters(): param.requires_grad = False
        num_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Linear(num_features, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    elif model_type == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
        for param in model.features[:-2].parameters(): param.requires_grad = False
        num_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.2), nn.Linear(num_features, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    else:
        raise ValueError(f"Modelo no soportado: {model_type}")
    return model

## 🚀 4. Lanzar Entrenamiento
Aquí puedes elegir si entrenar **ResNet50** o **MobileNetV2**, ajustar las épocas y el tamaño de lote (batch size).

In [ ]:
# ================= PARÁMETROS =================
MODEL_TYPE = "resnet50"  # Opciones: "resnet50" o "mobilenet_v2"
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 0.001
# ==============================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Ejecutando en: {device}")

dataset_dir = get_dataset_directory()
if dataset_dir is None:
    print("❌ ERROR CRÍTICO: No se puede proceder sin un dataset válido.")
else:
    image_paths, labels = load_dataset(dataset_dir)
    if not image_paths:
        print(f"❌ ERROR: No se encontraron imágenes válidas en {dataset_dir}.")
    else:
        train_transform = transforms.Compose([
            transforms.Resize((224, 224)), 
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15), 
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        val_transform = transforms.Compose([
            transforms.Resize((224, 224)), 
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        dataset = IncidentDataset(image_paths, labels, transform=train_transform)
        total = len(dataset)
        train_size = int(0.8 * total)
        val_size = int(0.1 * total)
        test_size = total - train_size - val_size
        
        train_data, val_data, test_data = random_split(
            dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
        )
        val_data.dataset.transform = val_transform
        test_data.dataset.transform = val_transform
        
        train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
        test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)
        
        model = create_model(MODEL_TYPE, len(CLASSES)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        
        best_val_acc = 0.0
        history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        
        model_save_path = MODELS_DIR / f"best_{MODEL_TYPE}.pt"
        
        print(f"\n⏱️ Iniciando Entrenamiento de {MODEL_TYPE} por {EPOCHS} épocas...")

        for epoch in range(EPOCHS):
            model.train()
            train_loss, correct, total_s = 0, 0, 0
            for images, targets, _ in tqdm(train_loader, desc=f"Época {epoch+1}/{EPOCHS} [Entrenar]", leave=False):
                images, targets = images.to(device), targets.to(device)
                outputs = model(images)
                loss = criterion(outputs, targets)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                _, pred = torch.max(outputs, 1)
                correct += (pred == targets).sum().item()
                total_s += targets.size(0)
            
            epoch_train_acc = correct / total_s
            epoch_train_loss = train_loss / len(train_loader)
            
            model.eval()
            val_loss, correct, total_s = 0, 0, 0
            with torch.no_grad():
                for images, targets, _ in val_loader:
                    images, targets = images.to(device), targets.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item()
                    _, pred = torch.max(outputs, 1)
                    correct += (pred == targets).sum().item()
                    total_s += targets.size(0)
            
            epoch_val_acc = correct / total_s
            epoch_val_loss = val_loss / len(val_loader)
            scheduler.step()
            
            history['train_loss'].append(epoch_train_loss)
            history['train_acc'].append(epoch_train_acc)
            history['val_loss'].append(epoch_val_loss)
            history['val_acc'].append(epoch_val_acc)
            
            print(f"Época {epoch+1:02d}: Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
            
            if epoch_val_acc > best_val_acc:
                best_val_acc = epoch_val_acc
                torch.save(model.state_dict(), model_save_path)
                print(f"  💾 ¡Nuevo mejor modelo guardado con precisión de validación: {best_val_acc:.4f}!")
                
        print(f"\n✅ Entrenamiento terminado. Mejor precisión de validación: {best_val_acc:.4f}")
        print(f"Modelo guardado en: {model_save_path}")

## 📊 5. Visualizar Resultados del Entrenamiento

In [ ]:
# Verificar si history fue definido (el entrenamiento no dio error)
if 'history' in locals() and history:
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Pérdida (Loss)')
    plt.xlabel('Época')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Val Accuracy')
    plt.title('Precisión (Accuracy)')
    plt.xlabel('Época')
    plt.legend()
    plt.show()
else:
    print("No se puede graficar: El entrenamiento no se ejecutó o falló.")

## 💾 6. Descargar el Modelo a tu Laptop
Ejecuta esta celda para descargar automáticamente el archivo del modelo entrenado (`best_resnet50.pt` o `best_mobilenet_v2.pt`) de vuelta a tu computadora para integrarlo en tu microservicio.

In [ ]:
from google.colab import files

best_model_file = f"models/best_{MODEL_TYPE}.pt"
if os.path.exists(best_model_file):
    print(f"Descargando {best_model_file}...")
    files.download(best_model_file)
else:
    print("El archivo del modelo no existe. Por favor, realiza el entrenamiento primero.")